# 📚 Student Score Prediction
### Linear Regression Model

Predicting student exam scores based on:
- **Study Hours** per day
- **Attendance %**
- **Sleep Hours** per day

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('student_dataset.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Basic Info:')
df.info()
print('\nStatistical Summary:')
df.describe().round(2)

In [ ]:
print('Missing Values:', df.isnull().sum().sum())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Score
plt.figure(figsize=(8, 4))
plt.hist(df['Score'], bins=30, color='steelblue', edgecolor='white')
plt.axvline(df['Score'].mean(), color='red', linestyle='--', label=f"Mean: {df['Score'].mean():.1f}")
plt.title('Distribution of Student Scores')
plt.xlabel('Score')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: each feature vs Score
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

features = ['Study_Hours', 'Attendance_Pct', 'Sleep_Hours']
colors   = ['#e74c3c', '#2ecc71', '#3498db']

for ax, feat, color in zip(axes, features, colors):
    ax.scatter(df[feat], df['Score'], alpha=0.3, color=color, edgecolors='none')
    # trend line
    z = np.polyfit(df[feat], df['Score'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, p(x_line), 'k--', linewidth=1.5)
    ax.set_xlabel(feat)
    ax.set_ylabel('Score')
    ax.set_title(f'{feat} vs Score')

plt.suptitle('Feature vs Score Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Prepare Data for Modeling

In [ ]:
X = df[['Study_Hours', 'Attendance_Pct', 'Sleep_Hours']]
y = df['Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

## 5. Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Model trained!')
print(f'\nIntercept: {model.intercept_:.2f}')
print('\nCoefficients:')
for feat, coef in zip(X.columns, model.coef_):
    print(f'  {feat:<20}: {coef:.4f}')

## 6. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('=== Model Performance ===')
print(f'MAE  (Mean Absolute Error) : {mae:.2f}')
print(f'MSE  (Mean Squared Error)  : {mse:.2f}')
print(f'RMSE (Root MSE)            : {rmse:.2f}')
print(f'R²   (R-squared)           : {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, color='steelblue', edgecolors='none')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Score')
axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f'Actual vs Predicted  (R² = {r2:.3f})')
axes[0].legend()

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4, color='#e67e22', edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.suptitle('Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Coefficient importance bar chart
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(coef_df['Feature'], coef_df['Coefficient'],
        color=['#2ecc71', '#3498db', '#e74c3c'], edgecolor='white')
plt.title('Feature Coefficients (Impact on Score)')
plt.ylabel('Coefficient Value')
plt.xlabel('Feature')
plt.tight_layout()
plt.show()

## 7. Predict a New Student's Score

In [ ]:
# Change these values to predict for any student
new_student = pd.DataFrame([{
    'Study_Hours':    6.0,
    'Attendance_Pct': 80.0,
    'Sleep_Hours':    7.0
}])

predicted_score = model.predict(new_student)[0]

print('Student Details:')
print(f'  Study Hours    : {new_student["Study_Hours"].values[0]}')
print(f'  Attendance %   : {new_student["Attendance_Pct"].values[0]}')
print(f'  Sleep Hours    : {new_student["Sleep_Hours"].values[0]}')
print(f'\nPredicted Score  : {predicted_score:.1f} / 100')